In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

csv_file_path = '/content/drive/MyDrive/2025-2026/156/WineQT.csv'
df = pd.read_csv(csv_file_path)

display(df.head())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Create a copy of the dataframe for feature engineering
df_fe = df.copy()

# 1. Total Acidity
df_fe['total_acidity'] = df_fe['fixed acidity'] + df_fe['volatile acidity']

# 2. Free / Total Sulfur Dioxide Ratio
if 'free sulfur dioxide' in df_fe.columns and 'total sulfur dioxide' in df_fe.columns:
    df_fe['sulfur_ratio'] = df_fe['free sulfur dioxide'] / (df_fe['total sulfur dioxide'] + 1e-5)

# 3. Alcohol to Sugar Ratio
if 'alcohol' in df_fe.columns and 'residual sugar' in df_fe.columns:
    df_fe['alcohol_sugar_ratio'] = df_fe['alcohol'] / (df_fe['residual sugar'] + 1e-5)

# 4. Sulphates to Chlorides Ratio (saltiness vs additives)
if 'sulphates' in df_fe.columns and 'chlorides' in df_fe.columns:
    df_fe['sulfate_chloride_ratio'] = df_fe['sulphates'] / (df_fe['chlorides'] + 1e-5)

# 5. pH and Acidity Interaction
if 'pH' in df_fe.columns:
    df_fe['ph_acidity'] = df_fe['pH'] * df_fe['fixed acidity']

# Prepare features (X) and target (y)
X_fe = df_fe.drop(columns=['quality', 'Id'], errors='ignore')
y_fe = df_fe['quality']

# Split the data into training and testing sets
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y_fe, test_size=0.2, random_state=42)

print("New features engineered. Here is a peek at the new dataset:")
display(X_fe.head())

New features engineered. Here is a peek at the new dataset:


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,total_acidity,sulfur_ratio,alcohol_sugar_ratio,sulfate_chloride_ratio,ph_acidity
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,8.10,0.323529,4.947342,7.367452,25.974
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,8.68,0.373134,3.769216,6.938068,24.960
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,8.56,0.277778,4.260851,7.064450,25.428
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,11.48,0.283333,5.157868,7.732302,35.392
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,8.10,0.323529,4.947342,7.367452,25.974


In [4]:
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Feature Engineering directly in the pipeline
df_fe = df.copy()
df_fe['total_acidity'] = df_fe['fixed acidity'] + df_fe['volatile acidity']
if 'free sulfur dioxide' in df_fe.columns and 'total sulfur dioxide' in df_fe.columns:
    df_fe['sulfur_ratio'] = df_fe['free sulfur dioxide'] / (df_fe['total sulfur dioxide'] + 1e-5)
if 'alcohol' in df_fe.columns and 'residual sugar' in df_fe.columns:
    df_fe['alcohol_sugar_ratio'] = df_fe['alcohol'] / (df_fe['residual sugar'] + 1e-5)
if 'sulphates' in df_fe.columns and 'chlorides' in df_fe.columns:
    df_fe['sulfate_chloride_ratio'] = df_fe['sulphates'] / (df_fe['chlorides'] + 1e-5)
if 'pH' in df_fe.columns:
    df_fe['ph_acidity'] = df_fe['pH'] * df_fe['fixed acidity']

# Prepare features (X) and target (y)
X = df_fe.drop(columns=['quality', 'Id'], errors='ignore')
y = df_fe['quality']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Best XGBoost model slightly tuned
xgb_best = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=150,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.8,
    random_state=42
)

# 2. Strong Random Forest model
rf_model = RandomForestRegressor(n_estimators=300, max_depth=15, max_features='sqrt', random_state=42)

# 3. ExtraTrees Regressor for added ensemble diversity
et_model = ExtraTreesRegressor(n_estimators=300, max_depth=15, max_features='sqrt', random_state=42)

# 4. HistGradientBoosting model (very fast and powerful)
hgb_model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=8, random_state=42)

# 5. Combine them using a Stacking Regressor
stacking_model = StackingRegressor(estimators=[
    ('xgb', xgb_best),
    ('rf', rf_model),
    ('et', et_model),
    ('hgb', hgb_model)
], final_estimator=RidgeCV())

# Train the ensemble
print("Training Stacking Ensemble Model (XGB + RF + ET + HGB) with FE...")
stacking_model.fit(X_train, y_train)

# Make predictions
y_pred = stacking_model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Stacking Ensemble Mean Squared Error (MSE): {mse:.4f}")
print(f"Stacking Ensemble R-squared (R2): {r2:.4f}")

Training Stacking Ensemble Model (XGB + RF + ET + HGB) with FE...
Stacking Ensemble Mean Squared Error (MSE): 0.2707
Stacking Ensemble R-squared (R2): 0.5135


In [6]:
from sklearn.feature_selection import RFECV
import matplotlib.pyplot as plt

# We'll use a slightly simplified XGBoost model to evaluate feature importance during RFE
estimator = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# Initialize RFECV (optimizing for negative MSE)
selector = RFECV(estimator, step=1, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

print("Performing Recursive Feature Elimination (RFECV)...")
selector = selector.fit(X_train, y_train)

print(f"\nOptimal number of features: {selector.n_features_}")

# Get the selected feature names
selected_features = X_train.columns[selector.support_]
print(f"Selected features: {list(selected_features)}")

# Transform the training and testing sets to keep only the selected features
X_train_rfe = selector.transform(X_train)
X_test_rfe = selector.transform(X_test)

# Retrain the Stacking Ensemble on the selected features
print("\nRetraining Stacking Ensemble on the selected features...")
stacking_model.fit(X_train_rfe, y_train)

# Make predictions and evaluate
y_pred_rfe = stacking_model.predict(X_test_rfe)
mse_rfe = mean_squared_error(y_test, y_pred_rfe)
r2_rfe = r2_score(y_test, y_pred_rfe)

print(f"\nRFE Stacking Ensemble Mean Squared Error (MSE): {mse_rfe:.4f}")
print(f"RFE Stacking Ensemble R-squared (R2): {r2_rfe:.4f}")

Performing Recursive Feature Elimination (RFECV)...

Optimal number of features: 15
Selected features: ['volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'total_acidity', 'sulfur_ratio', 'alcohol_sugar_ratio', 'sulfate_chloride_ratio', 'ph_acidity']

Retraining Stacking Ensemble on the selected features...

RFE Stacking Ensemble Mean Squared Error (MSE): 0.2707
RFE Stacking Ensemble R-squared (R2): 0.5136


In [7]:
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter grid for the stacking ensemble's base estimators
# The syntax uses 'estimatorname__parameter' to access base model params
param_grid_stacking = {
    'rf__n_estimators': [300, 400, 500],
    'rf__max_depth': [10, 15, 20, None],
    'et__n_estimators': [300, 400, 500],
    'et__max_depth': [10, 15, 20, None],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__max_depth': [4, 6, 8, 10],
    'hgb__learning_rate': [0.01, 0.05, 0.1],
    'hgb__max_iter': [100, 200, 300]
}

# Initialize RandomizedSearchCV
# Using cv=3 and n_iter=10 to keep execution time reasonable
random_search_stack = RandomizedSearchCV(
    estimator=stacking_model,
    param_distributions=param_grid_stacking,
    n_iter=10,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("Starting hyperparameter tuning for the Stacking Ensemble...")

# We fit using the RFE-selected features to match our best setup
random_search_stack.fit(X_train_rfe, y_train)

print(f"\nBest parameters found: {random_search_stack.best_params_}")

# Evaluate the best tuned model
best_tuned_stacking = random_search_stack.best_estimator_
y_pred_tuned_stack = best_tuned_stacking.predict(X_test_rfe)

mse_tuned_stack = mean_squared_error(y_test, y_pred_tuned_stack)
r2_tuned_stack = r2_score(y_test, y_pred_tuned_stack)

print(f"\n--- Tuned Stacking Ensemble Results ---")
print(f"Tuned Stacking Ensemble MSE: {mse_tuned_stack:.4f}")
print(f"Tuned Stacking Ensemble R2: {r2_tuned_stack:.4f}")
print(f"\nPrevious Best MSE: {mse_rfe:.4f}")

Starting hyperparameter tuning for the Stacking Ensemble...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


KeyboardInterrupt: 

In [8]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Predict on the training data using the best tuned stacking model
y_pred_train = best_tuned_stacking.predict(X_train_rfe)
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_pred_train)

print("--- Training Metrics ---")
print(f"Train MSE: {mse_train:.4f}")
print(f"Train RMSE: {rmse_train:.4f}")
print(f"Train R^2: {r2_train:.4f}")

# Calculate Test RMSE from previously calculated MSE
rmse_tuned_stack = np.sqrt(mse_tuned_stack)

print("\n--- Testing/Validation Metrics ---")
print(f"Test MSE: {mse_tuned_stack:.4f}")
print(f"Test RMSE: {rmse_tuned_stack:.4f}")
print(f"Test R^2: {r2_tuned_stack:.4f}")

--- Training Metrics ---
Train MSE: 0.0205
Train RMSE: 0.1432
Train R^2: 0.9695

--- Testing/Validation Metrics ---
Test MSE: 0.2668
Test RMSE: 0.5166
Test R^2: 0.5205


In [9]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

print("Evaluating individual base models from the Tuned Stacking Ensemble:")
print("-" * 50)

# Extract the fitted base estimators from the tuned stacking model
base_models = best_tuned_stacking.named_estimators_

for name, model in base_models.items():
    # Predict using the individual base model
    y_pred = model.predict(X_test_rfe)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f"Model: {name.upper()}")
    print(f"  Test MSE:  {mse:.4f}")
    print(f"  Test RMSE: {rmse:.4f}")
    print(f"  Test R^2:  {r2:.4f}")
    print("-" * 50)


Evaluating individual base models from the Tuned Stacking Ensemble:
--------------------------------------------------
Model: XGB
  Test MSE:  0.3080
  Test RMSE: 0.5550
  Test R^2:  0.4465
--------------------------------------------------
Model: RF
  Test MSE:  0.2879
  Test RMSE: 0.5366
  Test R^2:  0.4826
--------------------------------------------------
Model: ET
  Test MSE:  0.2671
  Test RMSE: 0.5168
  Test R^2:  0.5201
--------------------------------------------------
Model: HGB
  Test MSE:  0.3203
  Test RMSE: 0.5659
  Test R^2:  0.4244
--------------------------------------------------


In [10]:
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. Perform One-Hot Encoding
# (Note: Wine dataset is mostly numeric, but this handles any categorical variables)
df_encoded = pd.get_dummies(df_fe, drop_first=True)

# Prepare data with the encoded dataframe
X_enc = df_encoded.drop(columns=['quality', 'Id'], errors='ignore')
y_enc = df_encoded['quality']

# Split the data
X_train_enc, X_test_enc, y_train_enc, y_test_enc = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=42
)

# 2. Add Moderate L1 and L2 Regularization to base models to strike a balance

# XGBoost with moderate L1 (reg_alpha) and L2 (reg_lambda) regularization
xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=150,      # Increased to give model more capacity
    max_depth=6,           # Increased from 4 for better pattern recognition
    learning_rate=0.05,
    subsample=0.7,         # Slightly increased
    colsample_bytree=0.7,  # Slightly increased
    reg_lambda=1.0,        # Relaxed L2 Regularization
    reg_alpha=0.1,         # Relaxed L1 Regularization
    random_state=42
)

# Moderately Regularized Random Forest
rf_reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,          # Increased from 6
    min_samples_leaf=2,    # Reduced from 5
    max_features='sqrt',
    random_state=42
)

# Moderately Regularized Extra Trees
et_reg = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=12,          # Increased from 6
    min_samples_leaf=2,    # Reduced from 5
    max_features='sqrt',
    random_state=42
)

# HistGradientBoosting with Moderate L2 Regularization
hgb_reg = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.05,
    max_depth=6,           # Increased from 4
    l2_regularization=1.0, # Relaxed L2 penalty
    min_samples_leaf=5,    # Reduced from 10
    random_state=42
)

# Create the moderately regularized stacking ensemble
stacking_reg = StackingRegressor(estimators=[
    ('xgb', xgb_reg),
    ('rf', rf_reg),
    ('et', et_reg),
    ('hgb', hgb_reg)
], final_estimator=RidgeCV())

# 3. Train the moderately regularized model
print("Training Moderately Regularized Stacking Ensemble...")
stacking_reg.fit(X_train_enc, y_train_enc)

# 4. Evaluate Train vs Test to check for improved test performance
y_pred_train_reg = stacking_reg.predict(X_train_enc)
y_pred_test_reg = stacking_reg.predict(X_test_enc)

print("\n--- Moderately Regularized Model Metrics ---")
print(f"Train R^2: {r2_score(y_train_enc, y_pred_train_reg):.4f}")
print(f"Test R^2:  {r2_score(y_test_enc, y_pred_test_reg):.4f}")
print(f"Test MSE:  {mean_squared_error(y_test_enc, y_pred_test_reg):.4f}")
print("\nThe Test R^2 should be higher now by finding a better balance between underfitting and overfitting.")

Training Moderately Regularized Stacking Ensemble...

--- Moderately Regularized Model Metrics ---
Train R^2: 0.8883
Test R^2:  0.4930
Test MSE:  0.2821

The Test R^2 should be higher now by finding a better balance between underfitting and overfitting.


In [13]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("Training an even more strictly-tuned Random Forest Regressor...")

# Initialize a Random Forest with even stricter constraints
rf_single = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,            # Reduced further from 8 to 6
    min_samples_split=15,   # Increased from 10 to 15
    min_samples_leaf=10,    # Increased from 5 to 10
    max_features='sqrt',    # Still using a subset of features
    max_samples=0.7,        # Only use 70% of data per tree (Bootstrap subsampling)
    random_state=42
)

# Train the model on the encoded data
rf_single.fit(X_train_enc, y_train_enc)

# Make predictions
y_pred_train_rf = rf_single.predict(X_train_enc)
y_pred_test_rf = rf_single.predict(X_test_enc)

# Calculate metrics
print("\n--- Extremely Strict Random Forest Metrics ---")
print(f"Train R^2: {r2_score(y_train_enc, y_pred_train_rf):.4f}")
print(f"Test R^2:  {r2_score(y_test_enc, y_pred_test_rf):.4f}")
print(f"Test MSE:  {mean_squared_error(y_test_enc, y_pred_test_rf):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_enc, y_pred_test_rf)):.4f}")


Training an even more strictly-tuned Random Forest Regressor...

--- Extremely Strict Random Forest Metrics ---
Train R^2: 0.5173
Test R^2:  0.4034
Test MSE:  0.3320
Test RMSE: 0.5762


In [16]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# 1. Reframe the target variable for classification
# Quality >= 6 is Good (1), Quality < 6 is Bad (0)
y_class = (df_fe['quality'] >= 6).astype(int)

# 2. Split the data
# Using X_fe to avoid encoding other features as requested
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_fe, y_class, test_size=0.2, random_state=42
)

# 3. Initialize RandomForestClassifier with strict constraints
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=15,
    min_samples_leaf=10,
    max_features='sqrt',
    max_samples=0.7,
    random_state=42
)

# 4. Train the model
print("Training Strictly-Tuned Random Forest Classifier...")
rf_classifier.fit(X_train_cls, y_train_cls)

# 5. Evaluate the model
y_pred_train_cls = rf_classifier.predict(X_train_cls)
y_pred_test_cls = rf_classifier.predict(X_test_cls)

print("\n--- Classification Metrics ---")
print(f"Train Accuracy: {accuracy_score(y_train_cls, y_pred_train_cls):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test_cls, y_pred_test_cls):.4f}")
print(f"Train F1 Score: {f1_score(y_train_cls, y_pred_train_cls):.4f}")
print(f"Test F1 Score:  {f1_score(y_test_cls, y_pred_test_cls):.4f}")

print("\n--- Regression Metrics on New Encoded Binary Target ---")
print(f"Train R^2: {r2_score(y_train_cls, y_pred_train_cls):.4f}")
print(f"Test R^2:  {r2_score(y_test_cls, y_pred_test_cls):.4f}")
print(f"Train MSE: {mean_squared_error(y_train_cls, y_pred_train_cls):.4f}")
print(f"Test MSE:  {mean_squared_error(y_test_cls, y_pred_test_cls):.4f}")
print(f"Train RMSE:{np.sqrt(mean_squared_error(y_train_cls, y_pred_train_cls)):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_cls, y_pred_test_cls)):.4f}")

print("\n--- Classification Report (Test Data) ---")
print(classification_report(y_test_cls, y_pred_test_cls, target_names=['Bad Wine (<6)', 'Good Wine (>=6)']))

Training Strictly-Tuned Random Forest Classifier...

--- Classification Metrics ---
Train Accuracy: 0.8337
Test Accuracy:  0.7598
Train F1 Score: 0.8446
Test F1 Score:  0.7826

--- Regression Metrics on New Encoded Binary Target ---
Train R^2: 0.3304
Test R^2:  0.0277
Train MSE: 0.1663
Test MSE:  0.2402
Train RMSE:0.4078
Test RMSE: 0.4901

--- Classification Report (Test Data) ---
                 precision    recall  f1-score   support

  Bad Wine (<6)       0.73      0.74      0.73       102
Good Wine (>=6)       0.79      0.78      0.78       127

       accuracy                           0.76       229
      macro avg       0.76      0.76      0.76       229
   weighted avg       0.76      0.76      0.76       229

